# Medical-Chatbot Colab Evaluation (GenerationService)

Pipeline:
1. Load data/raw/VL_..._...json (VL merged dataset file)
2. Split by `q_type` (1, 2, 3)
3. Mode A (`LLM only`) generation with Qwen2.5-Instruct
4. Mode B (`LLM + RAG`) generation with Qwen2.5-Instruct + `chromadb/`
5. Evaluate with `src/evaluation/metrics.py`


In [ ]:
# ===== 0) clone repository (first run only) =====
!git clone -b hahyun https://github.com/ljhljh0703-cmd/Medical-Chatbot.git /content/Medical-Chatbot
%cd /content/Medical-Chatbot

In [ ]:
# ===== 1) Install dependencies (first run only) =====
!pip -q install -U pip
!pip -q install -r requirements.txt

In [ ]:
# ===== 2) Paths and runtime settings =====
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
SRC_PATH = REPO_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

DATA_PATH = REPO_ROOT / 'data/raw/VL_내과_통합.json'
CHROMA_PATH = REPO_ROOT / 'chroma_db'
OUTPUT_DIR = REPO_ROOT / 'outputs/colab_eval'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert DATA_PATH.exists(), f'Missing data file: {DATA_PATH}'
assert CHROMA_PATH.exists(), f'Missing ChromaDB path: {CHROMA_PATH}'
print('REPO_ROOT =', REPO_ROOT)
print('DATA_PATH =', DATA_PATH)
print('CHROMA_PATH =', CHROMA_PATH)

In [ ]:
# ===== 3) Load data and split by q_type =====
import json
import pandas as pd

rows = json.loads(DATA_PATH.read_text(encoding='utf-8'))
df = pd.DataFrame(rows)
df['q_type'] = df['q_type'].astype(int)

required_cols = ['qa_id', 'q_type', 'question', 'answer']
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f'Missing required column: {col}')

split_by_qtype = {q: g.reset_index(drop=True) for q, g in df.groupby('q_type')}
for q in [1, 2, 3]:
    print(f'q_type={q}:', len(split_by_qtype.get(q, pd.DataFrame())))

df.head(2)

In [ ]:
# ===== 4) Initialize GenerationService and RAG helpers =====
from config.settings import settings
from llm.generation_service import GenerationService
from retrieval.dense.embedder import Embedder
from retrieval.dense.chroma_store import ChromaStore

settings.model_backend = 'qwen'
settings.model_path = 'Qwen/Qwen2.5-7B-Instruct'
settings.model_mode = 'A'
settings.max_new_tokens = 256
settings.temperature = 0.2

# Retrieval embedding model.
# If OPENAI_API_KEY is missing, Embedder falls back to local sentence-transformers model.
settings.embedding_model = os.getenv('EMBEDDING_MODEL', 'jhgan/ko-sroberta-multitask')
settings.openai_api_key = os.getenv('OPENAI_API_KEY', settings.openai_api_key)

gen_service = GenerationService()
embedder = Embedder(embedding_model=settings.embedding_model, api_key=settings.openai_api_key)
store = ChromaStore(collection_name=settings.chroma_collection, persist_directory=str(CHROMA_PATH))

print('model_path =', settings.model_path)
print('embedding_model =', settings.embedding_model)
print('chroma_collection =', settings.chroma_collection)
print('chroma_count =', store.count())

In [ ]:
# ===== 5) Generation functions for Mode A / Mode B =====
from typing import Optional
from tqdm.auto import tqdm
import llm.generation_service as generation_service_module

QTYPE_SYSTEM_PROMPTS = {
    1: """당신은 한국 의학 국가고시 스타일 문제를 푸는 내과 전문의입니다.

[문항 유형]
- 객관식 5지선다 (q_type=1)

[답변 규칙]
- 반드시 보기 번호 1)~5) 중 하나만 선택해 답변합니다.
- 출력은 정답 한 줄만 작성합니다. 예: "3) 급성 심낭염"
- 이유, 해설, 인사말, 불필요한 부연 설명은 작성하지 않습니다.
- 문제에 제시된 선택지 밖의 답을 새로 만들지 않습니다.

[판단 기준]
- 증상, 병력, 활력징후, 검사 소견 등 임상 단서를 우선 반영해 가장 타당한 보기 하나를 고릅니다.
""",

    2: """당신은 내과 단답형 문항에 답하는 의학 어시스턴트입니다.

[문항 유형]
- 단답형 (q_type=2)

[답변 규칙]
- 정답 핵심어만 매우 간결하게 답합니다.
- 기본적으로 명사형(질환명, 약물명, 검사명, 해부학 용어)으로 작성합니다.
- 문장형 설명, 근거, 부연 설명은 작성하지 않습니다.
- 가능하면 1~5단어 이내로 답합니다.
- 필요할 때만 최소한의 괄호를 사용합니다.

[출력 예시 형식]
- "ACE 억제제"
- "아스피린"
- "나트륨"
""",

    3: """당신은 내과 서술형 문항에 답하는 전문의입니다.

[문항 유형]
- 서술형 (q_type=3)

[답변 규칙]
- 답변은 반드시 한국어만 사용합니다.
- 중국어(간체/번체), 일본어, 영어 문장으로 답하지 않습니다.
- 질문에서 요구한 항목을 빠짐없이 구조적으로 서술합니다.
- 임상적으로 중요한 기준(진단 기준, 중증도 기준, 치료 원칙/순서)을 명확히 포함합니다.
- 불필요하게 장황하지 않게, 핵심 위주로 문단 또는 번호를 사용해 정리합니다.
- 근거 없는 추측은 피하고, 제시된 정보와 일반적인 내과 원칙에 기반해 답합니다.
"""
}


def get_system_prompt_for_qtype(q_type: int) -> str:
    return QTYPE_SYSTEM_PROMPTS.get(int(q_type), QTYPE_SYSTEM_PROMPTS[3])


def build_rag_context(question: str, top_k: int = 5) -> str:
    q_vec = embedder.embed(question)
    results = store.query(query_embedding=q_vec, top_k=top_k)

    ids = (results.get('ids') or [[]])[0]
    docs = (results.get('documents') or [[]])[0]
    metas = (results.get('metadatas') or [[]])[0]
    dists = (results.get('distances') or [[]])[0]

    if not ids:
        return ''

    parts = []
    for i, (doc, meta, dist) in enumerate(zip(docs, metas, dists), 1):
        meta = meta or {}
        source = meta.get('source_spec', 'unknown')
        sim = max(0.0, 1.0 - (dist / 2.0))
        parts.append(f'[Ref {i}] (source: {source}, sim: {sim:.3f})\n{doc}')
    return '\n\n'.join(parts)


def generate_answer(question: str, q_type: int, mode: str = 'A', top_k: int = 5) -> str:
    system_prompt = get_system_prompt_for_qtype(q_type)
    original_prompt = generation_service_module.SYSTEM_PROMPT
    generation_service_module.SYSTEM_PROMPT = system_prompt
    try:
        if mode == 'A':
            return gen_service.generate(query=question, context=None, mode='A')
        if mode == 'B':
            ctx = build_rag_context(question, top_k=top_k)
            return gen_service.generate(query=question, context=ctx, mode='B')
        raise ValueError('mode must be A or B')
    finally:
        generation_service_module.SYSTEM_PROMPT = original_prompt


def run_generation(
    df_in,
    mode: str,
    q_type: int,
    top_k: int = 5,
    sample_n: Optional[int] = None,
    batch_size: int = 8,
):
    if batch_size <= 0:
        raise ValueError('batch_size must be >= 1')

    data = df_in.copy()
    if sample_n is not None:
        data = data.head(sample_n).copy()

    questions = data['question'].tolist()
    preds = []

    system_prompt = get_system_prompt_for_qtype(q_type)
    original_prompt = generation_service_module.SYSTEM_PROMPT
    generation_service_module.SYSTEM_PROMPT = system_prompt

    try:
        for i in tqdm(range(0, len(questions), batch_size), desc=f'Generate mode={mode}, q_type={q_type}'):
            batch_questions = questions[i : i + batch_size]

            if mode == 'A':
                batch_contexts = None
            elif mode == 'B':
                batch_contexts = [build_rag_context(q, top_k=top_k) for q in batch_questions]
            else:
                raise ValueError('mode must be A or B')

            try:
                batch_preds = gen_service.generate(
                    query=batch_questions,
                    context=batch_contexts,
                    mode=mode,
                )

                if isinstance(batch_preds, str):
                    batch_preds = [batch_preds]
                else:
                    batch_preds = list(batch_preds)

                if len(batch_preds) != len(batch_questions):
                    raise ValueError(
                        f'Batch output size mismatch: got {len(batch_preds)}, expected {len(batch_questions)}'
                    )

            except Exception:
                batch_preds = []
                for idx, q in enumerate(batch_questions):
                    try:
                        ctx = None if batch_contexts is None else batch_contexts[idx]
                        batch_preds.append(gen_service.generate(query=q, context=ctx, mode=mode))
                    except Exception as e:
                        batch_preds.append(f'[ERROR] {e}')

            preds.extend(batch_preds)
    finally:
        generation_service_module.SYSTEM_PROMPT = original_prompt

    out = data.copy()
    out[f'pred_{mode}'] = preds
    return out



In [ ]:
# ===== 6) Run Mode A and Mode B =====
# For quick checks in Colab, start with a small sample.
SAMPLE_N_PER_QTYPE = 20  # set None for full run
TOP_K = 5

mode_a_parts = []
mode_b_parts = []

for q in [1, 2, 3]:
    part = split_by_qtype.get(q)
    if part is None or len(part) == 0:
        continue

    print(f'q_type={q} prompt loaded')

    a_out = run_generation(part, mode='A', q_type=q, top_k=TOP_K, sample_n=SAMPLE_N_PER_QTYPE)
    b_out = run_generation(part, mode='B', q_type=q, top_k=TOP_K, sample_n=SAMPLE_N_PER_QTYPE)

    mode_a_parts.append(a_out[['qa_id', 'q_type', 'question', 'answer', 'pred_A']])
    mode_b_parts.append(b_out[['qa_id', 'q_type', 'pred_B']])

pred_a_df = pd.concat(mode_a_parts, ignore_index=True) if mode_a_parts else pd.DataFrame()
pred_b_df = pd.concat(mode_b_parts, ignore_index=True) if mode_b_parts else pd.DataFrame()

pred_df = pred_a_df.merge(pred_b_df, on=['qa_id', 'q_type'], how='left')
pred_df.to_csv(OUTPUT_DIR / 'predictions_A_B.csv', index=False, encoding='utf-8-sig')
print('saved:', OUTPUT_DIR / 'predictions_A_B.csv')
pred_df.head(3)



In [ ]:
# ===== 7) Evaluate with metrics.py =====
from evaluation.metrics import exact_match, rouge_l, bert_score

def evaluate_with_metrics(df_eval, pred_col: str):
    refs = df_eval['answer'].fillna('').tolist()
    preds = df_eval[pred_col].fillna('').tolist()

    ems = [exact_match(p, r) for p, r in zip(preds, refs)]
    rls = [rouge_l(p, r) for p, r in zip(preds, refs)]
    bss = bert_score(preds, refs)

    scored = df_eval.copy()
    scored['exact_match'] = ems
    scored['rouge_l'] = rls
    scored['bert_score'] = bss

    summary_rows = []
    for q in [1, 2, 3]:
        sub = scored[scored['q_type'] == q]
        if len(sub) == 0:
            continue
        summary_rows.append({
            'mode': pred_col.replace('pred_', ''),
            'q_type': q,
            'n': len(sub),
            'exact_match': float(sub['exact_match'].mean()),
            'rouge_l': float(sub['rouge_l'].mean()),
            'bert_score': float(sub['bert_score'].mean()),
        })

    summary_rows.append({
        'mode': pred_col.replace('pred_', ''),
        'q_type': 'all',
        'n': len(scored),
        'exact_match': float(scored['exact_match'].mean()),
        'rouge_l': float(scored['rouge_l'].mean()),
        'bert_score': float(scored['bert_score'].mean()),
    })

    return scored, pd.DataFrame(summary_rows)

scored_A, summary_A = evaluate_with_metrics(
    pred_df[['qa_id', 'q_type', 'question', 'answer', 'pred_A']].copy(),
    'pred_A'
)
scored_B, summary_B = evaluate_with_metrics(
    pred_df[['qa_id', 'q_type', 'question', 'answer', 'pred_B']].copy(),
    'pred_B'
)

summary_all = pd.concat([summary_A, summary_B], ignore_index=True)
summary_all.to_csv(OUTPUT_DIR / 'summary_metrics_A_B.csv', index=False, encoding='utf-8-sig')
scored_A.to_csv(OUTPUT_DIR / 'scored_mode_A.csv', index=False, encoding='utf-8-sig')
scored_B.to_csv(OUTPUT_DIR / 'scored_mode_B.csv', index=False, encoding='utf-8-sig')

print('saved:', OUTPUT_DIR / 'summary_metrics_A_B.csv')
print('saved:', OUTPUT_DIR / 'scored_mode_A.csv')
print('saved:', OUTPUT_DIR / 'scored_mode_B.csv')
summary_all

In [ ]:
# ===== 8) Download output files =====
from pathlib import Path
import shutil

# Download all files generated under output_dir in Google Colab
output_dir = Path("outputs/colab_eval")
zip_base = output_dir.parent / output_dir.name
zip_path = Path(shutil.make_archive(str(zip_base), "zip", root_dir=str(output_dir)))

from google.colab import files
files.download(str(zip_path))
print(f"Download ready: {zip_path}")

## Notes
- Set `SAMPLE_N_PER_QTYPE = None` for full evaluation.
- RAG quality depends on whether query embedding model matches the model used when building `chromadb/`.
- If Colab GPU memory is not enough, switch to a smaller Qwen model (for example, 3B).
